# 01 · 数据与合成遮挡

**本 notebook 做什么：** 从头到尾运行一遍会

1. 下载 LFW-funneled（仅第一次，约 230 MB，存放在 `data/`）；
2. 按身份做 70 / 15 / 15 的训练 / 验证 / 测试划分（**先划分原图，再做遮挡**，同一张原图不会出现在两个子集里）；
3. 为验证集和测试集生成**固定的**遮挡位置（独立随机种子），之后所有方法都在完全相同的遮挡图上评估；
4. 查看类别分布和遮挡效果。

所有结果保存在 `results/` 和 `figures/` 中，后面的 notebook 会复用同一份划分和遮挡。

> 注意：这是在"至少有 20 张照片的身份"上做的**闭集身份分类**，准确率**不等于** LFW 官方的人脸验证（verification）指标。

## 0. 环境检查
任何装好 `requirements.txt` 依赖的 Python 环境都可以（uv / venv / conda 均可，见 README）。
下面这个单元格会把仓库的 `src/` 加入 `sys.path`，所以**不需要**额外安装本项目的包；同时打印当前使用的 Python 路径，请确认它就是你装依赖的那个环境。

In [ ]:
%load_ext autoreload
%autoreload 2
# autoreload：修改 src/occprune 下的代码后无需重启 kernel

import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src" / "occprune").exists() else Path.cwd().parent  # 仓库根目录
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import occprune
print("Python  :", sys.executable)
print("occprune:", occprune.__file__)

## 1. 准备数据
`prepare_data` 是幂等的：如果该 seed 的划分、遮挡和统计文件已经存在就直接复用；设 `FORCE = True` 会重新生成（结果是确定性的，重新生成也完全一样）。

In [ ]:
from occprune.utils import load_config
from occprune.pipeline import prepare_data

cfg = load_config()     # 读取 configs/default.yaml；可以在这里修改 cfg[...] 做实验
SEED = cfg["seed"]      # 划分种子 = 训练种子
FORCE = False           # True = 重新生成划分和遮挡

stats = prepare_data(cfg, SEED, force=FORCE)
{k: v for k, v in stats.items() if k != "env"}   # 数据集统计：图片数、身份数、各子集大小、遮挡块边长等

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from occprune.utils import resolve
from occprune.data import load_lfw, load_split, class_counts
from occprune.occlusion import load_eval_boxes, apply_boxes_np, sample_train_boxes, patch_side

# 图片以 uint8 数组 (N, H, W, 3) 保存在内存中；数据集很小，不需要 DataLoader
d = load_lfw(cfg["data"], verbose=False)
images, labels, names = d["images"], d["labels"], d["target_names"]
split = load_split(SEED, labels)
print(images.shape, images.dtype, "identities:", len(names))
print({k: len(v) for k, v in split.items()})

## 2. 类别不平衡
最大的类别（George W Bush）约占全部图片的 17.5%，少数身份只有 20 张左右。
因此除了 accuracy，我们还报告 **balanced accuracy**（各类别召回率的平均），避免结果被大类主导。

In [ ]:
dist = pd.read_csv(resolve("results/class_distribution.csv"))   # 每个身份在各子集中的图片数
display(dist.head(8))

counts = class_counts(labels, len(names))
order = np.argsort(-counts)
plt.figure(figsize=(11, 3.2))
plt.bar(range(len(names)), counts[order])
plt.xlabel("identity (sorted)"); plt.ylabel("#images"); plt.title("Images per identity")
plt.show()
print(f"majority class: {names[order[0]]} = {counts[order[0]] / len(labels):.1%} of all images")

## 3. 训练时的随机遮挡（Dense-Aug 使用）
每张训练图以 0.5 的概率被遮挡：在随机位置贴一个黑色正方形，面积占整张图的 U(10%, 30%)。
遮挡在 **[0,1] 像素空间、ImageNet 归一化之前**进行（黑色 = 0）。下面随机抽 12 张训练图演示，标题是实际遮挡面积比例（0% 表示这张没被遮挡）。

In [ ]:
H, W = images.shape[1:3]
rng = np.random.default_rng(0)
idx = rng.choice(split["train"], 12, replace=False)
boxes = sample_train_boxes(len(idx), cfg["occlusion"]["train_prob"], tuple(cfg["occlusion"]["train_ratio"]), H, W, rng)
occ = apply_boxes_np(images[idx], boxes)   # boxes 每行是 [y0, x0, 边长]

fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for a, im, b, i in zip(axes.ravel(), occ, boxes, idx):
    a.imshow(im); a.axis("off")
    a.set_title(f"{names[labels[i]].split()[-1]}\n{b[2]**2/(H*W):.0%}", fontsize=8)
plt.tight_layout(); plt.show()

## 4. 测试集的固定遮挡
测试时有 4 个遮挡等级：0%、10%、30%、50%（占整张图面积的比例）。
每张测试图在每个遮挡等级下有 **3 个独立的固定遮挡位置**，结果取平均；所有方法都在这同一批遮挡图上评估，保证公平。

In [ ]:
test_boxes = load_eval_boxes("test", SEED)   # {遮挡比例: 数组 (重复次数, 测试图数, 3)}
for r, b in test_boxes.items():
    print(f"ratio {r:.2f}: shape {b.shape}, side {patch_side(r, H, W)} px")

i0 = split["test"][:4]   # 展示前 4 张测试图，第 1 组遮挡位置
fig, axes = plt.subplots(len(i0), len(test_boxes), figsize=(8, 8))
for col, (r, b) in enumerate(sorted(test_boxes.items())):
    ims = apply_boxes_np(images[i0], b[0][:len(i0)])
    for row in range(len(i0)):
        axes[row, col].imshow(ims[row]); axes[row, col].axis("off")
        if row == 0: axes[row, col].set_title("clean" if r == 0 else f"{int(r*100)}% occluded")
plt.tight_layout(); plt.show()